# User Input

In [1]:
team_url = '' # URL of your team i.e. 'https://team.celonis.cloud/'
api_key = '' 
key_type = 'USER_KEY' # USER_KEY or APP_KEY
data_pool_id = ''
data_model_id = ''

# Import Packages

In [2]:
from pycelonis import get_celonis
from pycelonis.pql import PQL, PQLColumn, PQLFilter, OrderByColumn
import pycelonis.pql as pql
from pycelonis.ems import ExportType
import pandas as pd
from tqdm import tqdm

# Load Celonis and Get Data Model

In [4]:
celonis = get_celonis(team_url, api_key, key_type)
data_pool = celonis.data_integration.get_data_pool(data_pool_id)
data_model = data_pool.get_data_model(data_model_id)

[2026-02-20 16:33:15,764] INFO: Initial connect successful! PyCelonis Version: 2.14.1
[2026-02-20 16:33:15,999] INFO: `ml-workbench` permissions: ['CREATE_APPS', 'USE_ALL_APPS', 'MANAGE_ALL_APPS', 'CREATE_WORKSPACES', 'MANAGE_ALL_WORKSPACES', 'VIEW_CONFIGURATION']
[2026-02-20 16:33:16,000] INFO: `team` permissions: ['MANAGE_AUDIT_LOGS', 'MANAGE_SSO_SETTINGS', 'USE_AUDIT_LOGS_API', 'TEAM_TO_TEAM_COPY', 'MANAGE_ADOPTION_VIEWS', 'MANAGE_GENERAL_SETTINGS', 'MANAGE_GROUPS', 'MANAGE_APPLICATIONS', 'MANAGE_ON_PREM_CLIENTS', 'USE_STUDIO_ADOPTION_API', 'MANAGE_LOGIN_HISTORY', 'MANAGE_LICENSE_SETTINGS', 'USE_LOGIN_HISTORY_API', 'USE_USER_GROUP_INFO_API', 'MANAGE_MEMBERS', 'MANAGE_UPLINK_INTEGRATIONS', 'MANAGE_PERMISSIONS', 'MANAGE_ADMIN_NOTIFICATIONS', 'MANAGE_DOWNLOAD_PORTAL', 'IMPORT_MEMBERS']
[2026-02-20 16:33:16,001] INFO: `process-repository` permissions: ['CREATE_AND_MODIFY_CATEGORIES', 'USE_CATEGORIES', 'DELETE_EXISTING_CATEGORIES', 'MODIFY_EXISTING_CATEGORIES']
[2026-02-20 16:33:16,001] 

In [12]:
skip_list = []

In [36]:
try:
    table_report_df = pd.read_csv('table_report.csv')
    table_report_row_list = table_report_df.to_dict(orient='records')
    new_report_flag = 0
except:
    new_report_flag = 1
    print('table_report.csv not found. Creating new table report...')
    table_report_df = {'Table':[], 'Field':[]}
    table_report_row_list = []

for table in tqdm(data_model.get_tables(),desc='Loading Tables...'):

    if (new_report_flag == 1 or table.name not in skip_list) and 't_e_' not in table.name:
        print(f'\n\nTable: {table.name}')

        for load_col in table.get_columns():

            if new_report_flag == 0:
                table_field_combos = (
                    table_report_df['Table'].astype(str) + '_' + table_report_df['Field'].astype(str)
                ).drop_duplicates().tolist()
            else:
                table_field_combos = []

            current_combo = table.name + '_' + load_col.name

            if current_combo not in table_field_combos:
                print(f'\nColumn: {load_col.name}')
                col = load_col.name
                load_col_list = [col]
                
                try:
                    col_df = pql.DataFrame({col: table.get_columns().find(col) for col in load_col_list}, data_model = data_model).to_pandas()

                    try:
                        col_type = load_col.type
                    except:
                        col_type = load_col.type_

                    col_report_row = {}
                    col_report_row['Table'] = table.name
                    col_report_row['Field'] = col
                    col_report_row['Data Type'] = col_type
                    c_mask = (col_df[col]
                        .dropna()
                        .astype(str)
                        .str.strip()
                        .ne(""))

                    if c_mask.empty:
                        col_report_row['Completeness'] = '0%'
                    else:
                        # col_report_row['Completeness'] = f"{round(100*(c_mask.sum()/len(c_mask)))}%"
                        col_report_row['Completeness'] = f"{round(100*(c_mask.sum()/len(col_df[col])),2)}%"
                        print(c_mask.sum())
                        print(len(col_df[col]))
                    col_report_row['Total Values'] = c_mask.sum()
                    col_report_row['Distinct Values'] = col_df[col].nunique(dropna=True)


                    non_null_col = col_df[col].dropna()
                    samples = []
                    for v in non_null_col:
                        if v not in samples:
                            samples.append(v)
                        if len(samples) == 5:
                            break

                    col_report_row['Sample Value 1'] = samples[0] if len(samples) > 0 else None
                    col_report_row['Sample Value 2'] = samples[1] if len(samples) > 1 else None
                    col_report_row['Sample Value 3'] = samples[2] if len(samples) > 2 else None
                    col_report_row['Sample Value 4'] = samples[3] if len(samples) > 3 else None
                    col_report_row['Sample Value 5'] = samples[4] if len(samples) > 4 else None
                    table_report_row_list.append(col_report_row)
                except:
                    print(f'ERROR - unable to load column {load_col}. Skipping...')

    new_report_flag = 0
    table_report_df = pd.DataFrame(table_report_row_list)
    table_report_df.to_csv('table_report.csv', index=False)

table_report.csv not found. Creating new table report...


Loading Tables...:   0%|          | 0/42 [00:00<?, ?it/s]



Table: t_o_custom_User

Column: TicketRestriction
[2026-02-20 17:47:28,260] INFO: Successfully created data export using api v1 with id '7fe88633-3908-4778-b981-a80c6322eb4e'
[2026-02-20 17:47:28,261] INFO: Wait for execution of data export with id '7fe88633-3908-4778-b981-a80c6322eb4e'


0it [00:00, ?it/s]

[2026-02-20 17:47:28,495] INFO: Export result chunks for data export with id '7fe88633-3908-4778-b981-a80c6322eb4e'
24908
24908

Column: Moderator
[2026-02-20 17:47:31,953] INFO: Successfully created data export using api v1 with id '6380e5ac-edf1-4c99-8fd9-ed4c35b9fe5c'
[2026-02-20 17:47:31,954] INFO: Wait for execution of data export with id '6380e5ac-edf1-4c99-8fd9-ed4c35b9fe5c'


...

[2026-02-20 18:03:57,366] INFO: Export result chunks for data export with id 'c6b5f8ba-e9ed-4ef2-b639-e2940df0625b'
677
677

Column: Case_ID
[2026-02-20 18:04:02,951] INFO: Successfully created data export using api v1 with id '7b800025-cb2c-44d2-86eb-453b8c892bae'
[2026-02-20 18:04:02,952] INFO: Wait for execution of data export with id '7b800025-cb2c-44d2-86eb-453b8c892bae'


0it [00:00, ?it/s]

[2026-02-20 18:04:03,174] INFO: Export result chunks for data export with id '7b800025-cb2c-44d2-86eb-453b8c892bae'


Loading Tables...: 100%|██████████| 42/42 [16:39<00:00, 23.81s/it]

658
677


In [35]:
't_e_' in table.name

True